# Prepare Norman data, train PDAE model and show evals

In [ ]:
%load_ext autoreload
%autoreload 2

# NOTE: workaround for relative imports in Jupyter notebooks
# This is necessary because the notebook is run from a different directory
# and the relative imports in the modules will fail otherwise.
import sys
sys.path.append("../")
sys.path.append("../../")

import random
from datetime import datetime
from pathlib import Path
import numpy as np
import torch
import anndata as ad
import pandas as pd
from scipy import sparse
from scipy.stats import gaussian_kde
import numpy as np
import matplotlib.pyplot as plt
from gdown import download as gdown_download
from PerturbationExtrapolation.pdae.api import PDAEData, PDAEConfig, PDAEModel

SEED = 2
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

## Preprocess according to Ahlman-Eltze et al. (2025)

We follow the experimental setup of Ahlman-Eltze et al. (https://www.nature.com/articles/s41592-025-02772-6):

- download the version of the Norman et al. (2019) data from scFoundation
- select the 1000 most highly variable genes in ctrl cells. 
- use half the double perturbations for training and the other half for evaluation, repeating this for 5 random splits.
- report the L2 distance between the predicted and ground truth mean gene expression.



In [ ]:
storage_path = Path("../data")
norman_adata_path = storage_path / "Norman2019_full.h5ad"
norman_adata_top1k_path = storage_path / "Norman2019_top1k.h5ad"

if not norman_adata_path.exists():
    # download version from scFoundation
    url = "https://figshare.com/ndownloader/files/44477939"
    gdown_download(url, str(norman_adata_path), quiet=False)


fraction_heldout_ctrl = 0.5  # fraction of control cells to hold out from training for evaluation
n_splits = 5  # number of random splits for double perturbations

if not norman_adata_top1k_path.exists():
    # if processed version with top 1000 highly variable genes and train-test-ood splits does not exist, create it
    adata = ad.read_h5ad(norman_adata_path)
    print("=== Norman et al. (2019) Data Overview ===")
    print(f"Number of genes: {len(adata.var_names)}")
    print(f"Number of cells: {len(adata.obs_names)}")

    # perturbations are encoded in the 'condition' column of adata.obs;
    # - the control condition is encoded as 'ctrl';
    # - single gene perturbations are encoded as 'GENE+crl' or 'crl+GENE'
    # - double perturbations are encoded as 'GENE1+GENE2'

    perturbation_labels = adata.obs['condition'].unique().tolist()
    control_count = 0
    other_count = 0

    # Track unique single genes and store perturbation conditions
    unique_single_genes = set()
    single_perts = []
    double_perts = []

    for condition in perturbation_labels:
        if condition == 'ctrl':
            control_count += 1
        else:
            # Split by '+' and filter out 'ctrl'
            genes = [g for g in condition.split('+') if g != 'ctrl']
            
            if len(genes) == 1:
                unique_single_genes.add(genes[0])
                single_perts.append(condition)
            elif len(genes) == 2:
                double_perts.append(condition)
            else:
                other_count += 1

    single_gene_count = len(unique_single_genes)
    double_gene_count = len(double_perts)

    print(f"Total experimental conditions: {len(perturbation_labels)}")
    print(f"Control conditions: {control_count}")
    print(f"Single gene perturbations (unique genes): {single_gene_count}")
    print(f"Double gene perturbations: {double_gene_count}")

    print("=== Filtering to Top 1000 highly expressed genes in CTRL condition ===")

    # Get control cells
    ctrl_mask = adata.obs['condition'] == 'ctrl'
    ctrl_cells = adata[ctrl_mask]
    print(f"Number of control cells: {ctrl_cells.n_obs}")

    # Calculate mean expression for each gene in control condition
    if sparse.issparse(ctrl_cells.X):
        ctrl_mean_expr = np.array(ctrl_cells.X.mean(axis=0)).flatten()
    else:
        ctrl_mean_expr = ctrl_cells.X.mean(axis=0)

    ctrl_mean_expr = pd.Series(ctrl_mean_expr, index=adata.var_names)
    top_1000_genes = ctrl_mean_expr.nlargest(1000).index.tolist()
    print(f"Original number of genes: {len(adata.var_names)}")
    print(f"Top 1000 gene mean expression range: {ctrl_mean_expr[top_1000_genes].min():.4f} - {ctrl_mean_expr[top_1000_genes].max():.4f}")

    # Filter adata to keep only these genes
    adata_top1k = adata[:, top_1000_genes].copy()
    print(f"Filtered number of genes: {len(adata_top1k.var_names)}")
    print(f"Number of cells (unchanged): {len(adata_top1k.obs_names)}")

    # Create multiple split columns for the experimental setup of Ahlmann-Eltze et al. (2023)
    print(f"=== Creating {n_splits} random splits ===")

    for split_idx in range(n_splits):
        split_name = f'splitAE{split_idx + 1}'
        print(f"\n--- Creating {split_name} ---")
        
        # Set seed for reproducibility of this split
        np.random.seed(SEED + split_idx)
        
        # Initialize the column
        adata_top1k.obs[split_name] = ''
        
        # 1. Assign all single gene perturbations to 'train'
        mask_single = adata_top1k.obs['condition'].isin(single_perts)
        adata_top1k.obs.loc[mask_single, split_name] = 'train'
        
        # 2. Assign control cells: fraction_heldout_ctrl to 'test', rest to 'train'
        ctrl_mask = adata_top1k.obs['condition'] == 'ctrl'
        ctrl_cells_idx = adata_top1k.obs[ctrl_mask].index
        n_test_ctrl = int(fraction_heldout_ctrl * len(ctrl_cells_idx))
        
        # Randomly select control cells for test
        test_ctrl_idx = np.random.choice(ctrl_cells_idx, size=n_test_ctrl, replace=False)
        
        adata_top1k.obs.loc[test_ctrl_idx, split_name] = 'test'
        adata_top1k.obs.loc[ctrl_mask & (adata_top1k.obs[split_name] == ''), split_name] = 'train'
        
        # 3. For double perturbations: randomly split in half
        double_perts_shuffled = double_perts.copy()
        np.random.shuffle(double_perts_shuffled)
        n_train_double = len(double_perts_shuffled) // 2
        
        train_double_perts = double_perts_shuffled[:n_train_double]
        ood_double_perts = double_perts_shuffled[n_train_double:]
        
        print(f"  Double perturbations assigned to 'train': {len(train_double_perts)}")
        print(f"  Double perturbations assigned to 'ood': {len(ood_double_perts)}")
        
        # Assign double perturbations using masks
        mask_train_double = adata_top1k.obs['condition'].isin(train_double_perts)
        mask_ood_double = adata_top1k.obs['condition'].isin(ood_double_perts)
        
        adata_top1k.obs.loc[mask_train_double, split_name] = 'train'
        adata_top1k.obs.loc[mask_ood_double, split_name] = 'ood'
        
        # Verify split distribution
        print(f"  Split distribution:")
        print(f"    {adata_top1k.obs[split_name].value_counts().to_dict()}")
        
        # Check for any unassigned cells
        unassigned = (adata_top1k.obs[split_name] == '').sum()
        if unassigned > 0:
            print(f"  Warning: {unassigned} cells were not assigned!")
        else:
            print(f"  All cells successfully assigned!")

    print("\n=== Summary of all splits ===")
    for split_idx in range(n_splits):
        split_name = f'splitAE{split_idx + 1}'
        print(f"{split_name}: {adata_top1k.obs[split_name].value_counts().to_dict()}")

    # save filtered AnnData (top 1k genes) to disk for later reuse
    adata_top1k.write_h5ad(norman_adata_top1k_path)
    print(f"\nSaved adata_top1k to {norman_adata_top1k_path}")

## Visualise data

In [ ]:
VISUALISE_DATA = False
if VISUALISE_DATA is True:
    # Select 12 random genes and  2 random single and double perturbations each create density plots
    selected_genes = np.random.choice(adata_top1k.var_names, size=12, replace=False)
    selected_single_perts = np.random.choice(single_perts, size=2, replace=False)
    selected_double_perts = np.random.choice(double_perts, size=2, replace=False)
    conditions_to_plot = ['ctrl'] + selected_single_perts.tolist() + selected_double_perts.tolist()

    # Create 3x4 subplot grid
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.flatten()
    colors = ['black', 'steelblue', 'forestgreen', 'coral', 'purple']
    labels = ['ctrl', 'S1', 'S2', 'D1', 'D2']
    # For each gene, create a density plot across the 5 conditions
    for idx, gene in enumerate(selected_genes):
        ax = axes[idx]
        gene_idx = np.where(adata_top1k.var_names == gene)[0][0]

        for condition, color, label in zip(conditions_to_plot, colors, labels):
            condition_mask = adata_top1k.obs['condition'] == condition
            condition_cells = adata_top1k[condition_mask]
            
            if sparse.issparse(condition_cells.X):
                gene_expr = condition_cells.X[:, gene_idx].toarray().flatten()
            else:
                gene_expr = condition_cells.X[:, gene_idx].flatten()
            
            if len(gene_expr) > 1:
                try:
                    kde = gaussian_kde(gene_expr)
                    x_range = np.linspace(gene_expr.min(), gene_expr.max(), 100)
                    linewidth = 3 if condition == 'ctrl' else 2
                    ax.plot(x_range, kde(x_range), color=color, label=label, linewidth=linewidth, alpha=0.8)
                except:
                    # If KDE fails, use histogram instead
                    ax.hist(gene_expr, bins=20, color=color, alpha=0.3, label=label, density=True)
        
        ax.set_title(gene, fontsize=11, fontweight='bold')
        ax.set_xlabel('Expression', fontsize=9)
        ax.set_ylabel('Density', fontsize=9)
        ax.legend(fontsize=7, loc='upper right')
        ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(storage_path / 'gene_expression_densities.png', dpi=150, bbox_inches='tight')
    plt.show()

### PDAEData

In [ ]:
data_name = "Norman_AE(top1k)"
pdae_data_path = storage_path / f"pdae_data_{data_name}.pth"

if pdae_data_path.exists():
    pdae_data = PDAEData.load(file_path=pdae_data_path)

else:
    pdae_data = PDAEData.from_adata(
        # adata_path=norman_adata_path,
        adata_path=norman_adata_top1k_path,
        pert_col="condition",
        pert_sep="+",
        pert_ctrl="ctrl",
        split_col="splitAE",
        # split_col=f"split{norman_split}",
        train_split="train",
        train_holdout_split="test", # NOTE: the choice for these names comes from https://github.com/facebookresearch/CPA/blob/main/preprocessing/Norman19.ipynb
        val_split=None,
        test_split="ood",
        data_dir=storage_path,
        data_name=data_name
    )

### PDAEConfig

In [ ]:
current_time = datetime.now().strftime('%Y-%m-%d_%H-%M')
pdae_save_dir = storage_path / str(current_time + "_norman") / "norman_split_AE"
pdae_save_dir.mkdir(parents=True, exist_ok=True)

model_name = 'norman'

pdae_config = PDAEConfig(**{
    'device': 'cuda:0' if torch.cuda.is_available() else 'cpu',
    'seed': SEED,
    'model_name': model_name,
    'batch_size': 128,
    'learning_rate': 1e-4,
    'dim_z_model': 101,
    'dim_noise_model': 27,
    'sigma_noise_model': 0.1,
    'decoder_layer_shapes': [128, 256, 256, 512],
    'encoder_layer_shapes': [512, 256, 256, 128],
    'weight_reconstruction_loss': 0.1,
    'weight_perturbation_energy_loss': 1.0,
    'weight_marginal_prior_energy_loss': 0.001,
    'weight_l21': 0.001,
    'update_encoder_on_reconstruction_loss': False,
    'normalize_energy_loss': False,
    'beta': 1.0,
    'use_bias_in_perturbation_matrix': False,
    'use_softplus_after_decoder': True,
})

### PDAEModel

In [ ]:
pdae = PDAEModel("cuda:0", pdae_data, pdae_config, save_dir=pdae_save_dir)

### Training

In [ ]:
N_EPOCHS = 1000

pdae.train(
    num_epochs=N_EPOCHS,
    n_epochs_model_checkpoint=N_EPOCHS,
    n_epochs_plot_losses=50,
    # losses_to_plot=["tr_reconstruction_losses", "tr_perturbation_energy_losses", "val_energy_losses", "val_mean_diffs"],
)

### Evaluate the model and benchmark

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

from PerturbationExtrapolation.pdae.baselines import get_baselines
from PerturbationExtrapolation.pdae.metrics import get_metrics, MeanDifferenceMetric

pdae.run_eval(
    models=get_baselines(),
    # metrics=get_metrics(),
    metrics=[MeanDifferenceMetric()],
    metrics_device="cuda:0",
    use_train_holdout=False,
    ratio_subsample_train=1,  
    ratio_subsample_val_test=1,
    run_eval_batched=False,
    return_df=True,
)

df_evals_agg = pdae.get_agg_eval_results(layout="wide")
df_evals_agg